<a href="https://colab.research.google.com/github/habibiputrar/BigData26_A_2411531001_Habibi-Putra-Rizqullah-/blob/main/Praktikum05/BD_A_TugasP05_2411531001_Habibi_Putra_Rizqullah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PRAKTIKUM BIG DATA 5
## Tugas : Inverted Index dengan MapReduce

**Nama:** Habibi Putra Rizqullah  
**NIM:** 2411531001  
**Kelas:** A


## 1. Persiapan

In [ ]:
from pathlib import Path
import collections, hashlib, json, re, shutil
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

KANDIDAT = [
    Path('lapisan_terkurasi'),
    Path('/content/drive/MyDrive/BigData/Praktikum5/lapisan_terkurasi'),
]
TERKURASI = next((p for p in KANDIDAT if p.exists()), None)
if TERKURASI is None:
    raise FileNotFoundError('lapisan_terkurasi tidak ditemukan. Jalankan notebook utama sampai H-6.')

OUT = Path('tugas_mandiri')
OUT.mkdir(exist_ok=True)
print('Sumber data:', TERKURASI.resolve())

Mounted at /content/drive
Sumber data: /content/drive/MyDrive/BigData/Praktikum5/lapisan_terkurasi


## 2. Baca data

Tiap file Parquet berisi satu kategori, dan kolom `kategori` diambil dari nama foldernya (`kategori=news`, dst).

In [ ]:
dok = pd.read_parquet(TERKURASI, columns=['judul', 'kategori', 'tanggal', 'id', 'url'])
dok['kategori'] = dok['kategori'].astype(str)
dok['tanggal'] = pd.to_datetime(dok['tanggal'])

print('Jumlah judul   :', f'{len(dok):,}')
print('ID sumber kosong:', f"{int(dok['id'].isna().sum()):,}", '(semuanya kategori inet)')
print(dok.loc[dok['id'].isna(), 'kategori'].value_counts().to_dict())

Jumlah judul   : 90,818
ID sumber kosong: 5,577 (semuanya kategori inet)
{'inet': 5577}


## 3. ID dokumen

Kolom `id` kosong untuk semua judul kategori inet karena URL aslinya rusak (`#`). Kalau indeks langsung memakai kolom itu, 5.577 judul inet tidak ikut terindeks.

Jadi tiap judul saya beri `doc_id` sendiri, yaitu hash SHA-256 dari kategori, tanggal, judul, dan id sumber (kalau ada). Datanya sama berarti ID-nya sama, jadi hasilnya selalu bisa diulang. Id sumber yang asli tetap disimpan di tabel pemetaan supaya bisa dilacak.

In [ ]:
def buat_doc_id(kategori, tanggal, judul, id_sumber):
    bahan = [kategori, str(tanggal.date()), judul, None if pd.isna(id_sumber) else int(id_sumber)]
    return 'doc_' + hashlib.sha256(json.dumps(bahan, ensure_ascii=False).encode()).hexdigest()

dok['doc_id'] = [
    buat_doc_id(r.kategori, r.tanggal, r.judul, r.id)
    for r in dok.itertuples(index=False)
]
assert dok['doc_id'].is_unique

pemetaan = dok[['doc_id', 'id', 'judul', 'kategori', 'tanggal', 'url']]
pemetaan.to_parquet(OUT / 'pemetaan_dokumen.parquet', index=False)
print('Dokumen:', f'{len(dok):,}', '| contoh ID:', dok['doc_id'].iloc[0][:20] + '...')

Dokumen: 90,818 | contoh ID: doc_b3323964176b9bd8...


## 4. Map, shuffle, reduce

Map mengeluarkan pasangan `(kata, doc_id)` untuk tiap judul, dan kata yang muncul dua kali di satu judul hanya dihitung sekali. Shuffle mengumpulkan `doc_id` berdasarkan kata. Reduce mengurutkan daftar `doc_id` tiap kata.

Satu kategori jadi satu input split, sama seperti H-9. Stopword tidak dibuang karena indeks ini harus bisa dipakai untuk mencari kata apa saja.

In [ ]:
def tokenisasi(teks):
    return re.findall(r'[a-z]+', teks.lower())

def mapper_index(df):
    for r in df.itertuples(index=False):
        for kata in set(tokenisasi(r.judul)):
            yield kata, r.doc_id

def shuffle_index(pasangan):
    kelompok = collections.defaultdict(set)
    for kata, doc_id in pasangan:
        kelompok[kata].add(doc_id)
    return kelompok

def reducer_index(kata, ids):
    return kata, sorted(ids)

kelompok = collections.defaultdict(set)
for kategori, grup in dok.groupby('kategori'):
    for kata, ids in shuffle_index(mapper_index(grup)).items():
        kelompok[kata].update(ids)

inverted_index = dict(reducer_index(k, ids) for k, ids in kelompok.items())
print('Kata unik:', f'{len(inverted_index):,}')

Kata unik: 32,275


## 5. Hasil dan pengecekan

In [ ]:
index_df = pd.DataFrame(
    [{'kata': k, 'doc_ids': v, 'jumlah_dokumen': len(v)} for k, v in sorted(inverted_index.items())]
)
index_df.to_parquet(OUT / 'inverted_index.parquet', index=False)

contoh = [
    {'kata': k,
     'jumlah_dokumen': len(inverted_index.get(k, [])),
     'lima_id_pertama': inverted_index.get(k, [])[:5]}
    for k in ['corona', 'virus', 'mobil']
]
(OUT / 'contoh_inverted_index.json').write_text(
    json.dumps(contoh, ensure_ascii=False, indent=2), encoding='utf-8')

for c in contoh:
    print(c['kata'], '->', f"{c['jumlah_dokumen']:,}", 'dokumen')
    for i in c['lima_id_pertama']:
        print('   ', i)

corona -> 17,228 dokumen
    doc_0002318159cf336480bf3386107380442996263df59655bce093108ebba9ce73
    doc_00061f39dd13ea855209657cd9e9273dbdf354cfb42f078ef5dcb095e548a1e2
    doc_0008b8de9b3997146f912ae6b8ac84004703bf2618aa5419e679156cebc6a339
    doc_000abe949ad3d9789ef5c9987653b50d9cee6e0676b0d40f5bf68d2d9b1eee76
    doc_000d4964b551a9c2c49b8ffd152b6f1cacbdf2f1a86aaafe12a04776ea978b6c
virus -> 3,067 dokumen
    doc_0008b8de9b3997146f912ae6b8ac84004703bf2618aa5419e679156cebc6a339
    doc_001116073cbd779f78a0a07b4e5de2dabd086dc0c7637a79eaf7237df96a7c43
    doc_001a882a64fafedbef51a4ad97c4e98ae2e63ada07b951ed4bfd72223a495710
    doc_002e2f78acbbe87763139157e434ce95720f1d795e726c50cb4499be6aa0abb2
    doc_0036c182208eafc2971587e33048cd89fa76eeb56cbdc8adf977f3146ebab634
mobil -> 1,678 dokumen
    doc_0005732ac648fa985f05e78e158c1b33d74cadffb9caecb261d4f6f156bfdcb0
    doc_0012501180b57eec1c45633d7af506d20036d8613f2665cdab6af014251b88fa
    doc_00272e3349c0e60a91703375e1968cfbc2e4b6d484a08

In [ ]:
# cek silang: hitung langsung dari judul tanpa MapReduce
for kata in ['corona', 'virus', 'mobil']:
    langsung = int(dok['judul'].map(lambda t: kata in set(tokenisasi(t))).sum())
    print(f'{kata:<8} index={len(inverted_index[kata]):>6,} | langsung={langsung:>6,} | sama={len(inverted_index[kata]) == langsung}')

assert all(len(ids) == len(set(ids)) for ids in inverted_index.values())
print('\nKata unik:', f'{len(inverted_index):,}', '| Dokumen:', f'{len(dok):,}',
      '| Judul tanpa ID sumber:', f"{int(dok['id'].isna().sum()):,}")

corona   index=17,228 | langsung=17,228 | sama=True
virus    index= 3,067 | langsung= 3,067 | sama=True
mobil    index= 1,678 | langsung= 1,678 | sama=True

Kata unik: 32,275 | Dokumen: 90,818 | Judul tanpa ID sumber: 5,577


## 6. Simpan hasil

In [ ]:
TUJUAN_DRIVE = Path('/content/drive/MyDrive/BigData/Praktikum5/Tugas Praktikum')
if Path('/content/drive/MyDrive').exists():
    TUJUAN_DRIVE.mkdir(parents=True, exist_ok=True)
    shutil.copytree(OUT, TUJUAN_DRIVE, dirs_exist_ok=True)
    print('Disalin ke', TUJUAN_DRIVE)
print(sorted(p.name for p in OUT.iterdir()))

Disalin ke /content/drive/MyDrive/BigData/Praktikum5/Tugas Praktikum
['contoh_inverted_index.json', 'inverted_index.parquet', 'pemetaan_dokumen.parquet']
